## Runnable RAG chain:
``` python
                              User Question
                                   ↓
                              Retriever (Runnable)
                                   ↓
                              Context Formatter
                                   ↓
                              Prompt Template
                                   ↓
                                  LLM
                                   ↓
                                 Answer
```

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Text Loader
loader = TextLoader('E:\\SSPL_Internship_Repo\\Jan_12_25\\RAG_tutorial\\data\\sample.txt')
documents = loader.load()

# Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

# Embeddings and Vector Store
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Retriever with similarity search
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


e:\SSPL_Internship_Repo\Jan_12_25\RAG_tutorial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Dell\AppData\Local\Temp\ipykernel_16340\1973154569.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

The prompt explicitly tells the LLM:

    - Use ONLY retrieved context
    - Don’t hallucinate

In [3]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not present, say "I don't know".

Context:
{context}

Question:
{question}
"""
)

In [ ]:
# Retriever returns List[Document], LLM expects string context.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [9]:
# LLM 
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="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",
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_tokens=512
)

CHAIN LOGIC :

- Question goes to retriever
- Retrieved docs → formatted context
- Question passes through unchanged
- Prompt → LLM → clean text output

In [10]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
answer = rag_chain.invoke(
    "What is this document about?"
)
print(answer)

This document is about AI Agents, specifically their overview and applications.


In [13]:
answer = rag_chain.invoke(
    "what are the Key Characteristics of AI Agents?"
)
print(answer)

The Key Characteristics of AI Agents are:
1. Autonomy
2. Reactivity
3. Proactivity
4. Social Ability
